# Perceptron: Yapay Sinir Ağlarının Temeli

Bu notebook, yüksek lisans dersinde anlatım amacıyla hazırlanmıştır.

İçerik akışı:
- Perceptron nedir?
- Matematiksel model
- Aktivasyon fonksiyonu
- AND problemi
- Ağırlık güncelleme mantığı
- Python ile perceptron kodlama
- Perceptronun sınırları
- XOR problemi ve çok katmanlı ağlara geçiş


## 1. Perceptron Nedir?

Perceptron, en basit yapay sinir ağı modelidir.

Temel mantık:
- Girdiler alınır
- Her giriş bir ağırlık ile çarpılır
- Toplam değer hesaplanır
- Bir aktivasyon fonksiyonu ile çıktı üretilir

Yani perceptron, temelde bir **ikili sınıflandırıcıdır**.


## 2. Matematiksel Model

Perceptronun matematiksel modeli şu şekildedir:

$$y = f(x_1 w_1 + x_2 w_2 + \dots + x_n w_n + b)$$

Burada:
- $x_i$: girişler
- $w_i$: ağırlıklar
- $b$: bias
- $f$: aktivasyon fonksiyonu


## 3. Aktivasyon Fonksiyonu

Bu örnekte klasik **step function** kullanacağız:

$$
f(x) = 
\begin{cases}
1, & x \geq 0 \\
0, & x < 0
\end{cases}
$$

Yani toplam değer sıfır veya daha büyükse çıktı 1, değilse 0 olur.


## 4. Problemimiz: AND Mantık Kapısı

Perceptronun öğrenmesini göstermek için AND problemini kullanacağız.

| x1 | x2 | çıktı |
|----|----|-------|
| 0  | 0  | 0     |
| 0  | 1  | 0     |
| 1  | 0  | 0     |
| 1  | 1  | 1     |

Bu problem **doğrusal olarak ayrılabilir** olduğu için perceptron ile çözülebilir.


## 5. Gerekli Veri Kümesini Tanımlayalım


In [ ]:
X = [
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
]

y = [0, 0, 0, 1]

print('Girdi verileri:', X)
print('Hedef çıktılar:', y)


## 6. Perceptron Sınıfını Yazalım

Şimdi perceptronu sıfırdan kodlayacağız.

Bu sınıfta şunlar olacak:
- başlangıç ağırlıkları
- bias
- aktivasyon fonksiyonu
- tahmin fonksiyonu
- eğitim fonksiyonu


In [ ]:
import random

class Perceptron:
    def __init__(self, input_size, learning_rate=0.1):
        self.weights = [random.uniform(-1, 1) for _ in range(input_size)]
        self.bias = random.uniform(-1, 1)
        self.learning_rate = learning_rate

    def activation(self, x):
        return 1 if x >= 0 else 0

    def predict(self, inputs):
        total = 0
        for i in range(len(inputs)):
            total += inputs[i] * self.weights[i]
        total += self.bias
        return self.activation(total)

    def train(self, training_data, labels, epochs=10):
        for epoch in range(epochs):
            print(f'\nEpoch {epoch+1}')
            for inputs, target in zip(training_data, labels):
                prediction = self.predict(inputs)
                error = target - prediction

                for i in range(len(self.weights)):
                    self.weights[i] += self.learning_rate * error * inputs[i]

                self.bias += self.learning_rate * error

                print(f'Girdi: {inputs}, Hedef: {target}, Tahmin: {prediction}, Hata: {error}')
                print(f'Güncel weights: {self.weights}, bias: {self.bias}')


## 7. Kodun Adım Adım Açıklaması

### `__init__`
Bu bölümde perceptronun başlangıç değerleri tanımlanır:
- Her giriş için rastgele bir ağırlık atanır.
- Bias değeri rastgele başlatılır.
- Öğrenme katsayısı belirlenir.

### `activation`
Bu fonksiyon step function'dır.

### `predict`
Bu fonksiyon perceptronun ileri besleme kısmıdır:
1. Girdiler ile ağırlıkları çarpar
2. Hepsini toplar
3. Bias ekler
4. Aktivasyon fonksiyonunu uygular

### `train`
Bu fonksiyon öğrenmeyi gerçekleştirir:
1. Tahmin yapar
2. Hata hesaplar
3. Ağırlıkları günceller
4. Bias'ı günceller


## 8. Ağırlık Güncelleme Kuralı

Perceptron öğrenme kuralı:

$$w_i = w_i + \eta \cdot error \cdot x_i$$

$$b = b + \eta \cdot error$$

Burada:
- $\eta$: learning rate
- `error = target - prediction`

Eğer hata 0 ise model doğru tahmin yapmıştır ve güncelleme yapılmaz.


## 9. Perceptronu Oluşturup Eğitelim


In [ ]:
p = Perceptron(input_size=2, learning_rate=0.1)
print('Başlangıç weights:', p.weights)
print('Başlangıç bias:', p.bias)

p.train(X, y, epochs=10)


## 10. Eğitilen Modeli Test Edelim


In [ ]:
print('\nTest Sonuçları:')
for inputs in X:
    print(f'{inputs} -> {p.predict(inputs)}')


## 11. Elle Bir Güncelleme Örneği

Diyelim ki:
- giriş = `[1, 1]`
- hedef = `1`
- tahmin = `0`
- hata = `1`
- learning rate = `0.1`

Başlangıçta:
- $w_1 = 0.2$
- $w_2 = -0.1$
- $b = 0.0$

Güncelleme sonrası:

$$w_1 = 0.2 + 0.1 \cdot 1 \cdot 1 = 0.3$$
$$w_2 = -0.1 + 0.1 \cdot 1 \cdot 1 = 0.0$$
$$b = 0.0 + 0.1 \cdot 1 = 0.1$$

Yani model, bu örneği gelecekte daha kolay 1 sınıfına atayacak şekilde kendini düzeltir.


## 12. Epoch Kavramı

- 1 epoch = tüm eğitim verisinin baştan sona bir kez işlenmesi
- Epoch sayısı arttıkça modelin doğru sınıflandırma ihtimali artar

Ancak perceptronun başarısı, problemin lineer ayrılabilir olmasına bağlıdır.


## 13. Perceptronun Sınırı

Perceptron sadece **doğrusal olarak ayrılabilen** problemleri çözebilir.

Çözer:
- AND
- OR

Çözemez:
- XOR


## 14. XOR Problemi

| x1 | x2 | çıktı |
|----|----|-------|
| 0  | 0  | 0     |
| 0  | 1  | 1     |
| 1  | 0  | 1     |
| 1  | 1  | 0     |

Bu veri tek bir doğru ile ayrılamaz.
Bu yüzden tek katmanlı perceptron burada başarısız olur.


## 15. Sonuç

- Perceptron, yapay sinir ağlarının temel yapı taşıdır.
- Tek nöronlu basit bir modeldir.
- Doğrusal sınıflandırma yapabilir.
- Daha karmaşık problemler için çok katmanlı yapay sinir ağlarına ihtiyaç vardır.

Geçiş cümlesi:

> Tek perceptron bazı problemleri çözebilir; ancak görüntü işleme gibi karmaşık alanlarda çok katmanlı yapılara ihtiyaç vardır.


## 16. İsteğe Bağlı: Daha Sade Sürüm

Aşağıda sınıf yapısı olmadan, daha düz mantıkla yazılmış perceptron örneği verilmektedir. Bu sürüm derste ilk anlatım için bazen daha kolay olabilir.


In [ ]:
import random

w1 = random.uniform(-1, 1)
w2 = random.uniform(-1, 1)
b = random.uniform(-1, 1)
lr = 0.1

X = [[0, 0], [0, 1], [1, 0], [1, 1]]
y = [0, 0, 0, 1]

def step(x):
    return 1 if x >= 0 else 0

for epoch in range(10):
    print(f'\nEpoch {epoch+1}')
    for i in range(len(X)):
        x1, x2 = X[i]
        target = y[i]

        total = x1 * w1 + x2 * w2 + b
        prediction = step(total)
        error = target - prediction

        w1 = w1 + lr * error * x1
        w2 = w2 + lr * error * x2
        b = b + lr * error

        print(f'Girdi: {[x1, x2]}, Hedef: {target}, Tahmin: {prediction}, Hata: {error}')

print('\nSon test:')
for i in range(len(X)):
    x1, x2 = X[i]
    total = x1 * w1 + x2 * w2 + b
    prediction = step(total)
    print(f'{[x1, x2]} -> {prediction}')


Öğrencilerin vize, final ve ödev notlarını kullanarak benzer bir perceptron modeli oluşturup, bu modelin doğruluk oranını hesaplamalarını yapınız.